# Task 2: Citation Mapping - BERT Training

**Model:** bert-base-uncased (Sequence Classification)

**Bài toán:** Cho text có [CITATION_X] + danh sách candidate papers (title + abstract) → Predict [CITATION_X] map với paper nào?

**Approach:** Với mỗi cặp (context, candidate paper) → model predict match (1) hoặc không match (0)

**Dataset:** task2-citation-mapping

---

## 1. Setup & Imports

In [ ]:
import transformers, datasets, accelerate
print(f"✅ transformers: {transformers.__version__}")
print(f"✅ datasets: {datasets.__version__}")
print(f"✅ accelerate: {accelerate.__version__}")

## 2. Wandb Login

In [ ]:
import wandb
import os

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    key = secrets.get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = key
    wandb.login(key=key, relogin=True)
    print("✅ Wandb logged in")
except Exception as e:
    print(f"⚠️ Wandb login failed: {e}")
    os.environ["WANDB_MODE"] = "disabled"
    print("⚠️ Wandb disabled — training will continue without logging")

## 3. Data Paths

In [ ]:
import os

DATA_ROOT = "/kaggle/input/datasets/tathiyennhi/task2-citation-mapping/task2"

train_path = os.path.join(DATA_ROOT, "train")
val_path = os.path.join(DATA_ROOT, "val")

# Uncomment nếu cần xem cấu trúc thư mục
# for root, dirs, files in os.walk("/kaggle/input"):
#     print(root, dirs[:5], files[:5])

train_count = len([f for f in os.listdir(train_path) if f.endswith('.label')])
val_count = len([f for f in os.listdir(val_path) if f.endswith('.label')])

print(f"✅ Train: {train_count:,} files")
print(f"✅ Val: {val_count:,} files")

In [ ]:
import json
from pathlib import Path

sample_label = sorted(Path(train_path).glob("*.label"))[0]
sample_in = sample_label.with_suffix(".in")

print("=== FILE .in ===")
with open(sample_in) as f:
    in_data = json.load(f)
print(f"Keys: {list(in_data.keys())}")
print(f"Text (first 200): {in_data.get('text', '')[:200]}")
print(f"Candidates: {len(in_data.get('citation_candidates', []))}")
print(f"Bib entries: {len(in_data.get('bib_entries', {}))}")

print("\n=== FILE .label ===")
with open(sample_label) as f:
    label_data = json.load(f)
print(f"Keys: {list(label_data.keys())}")
print(f"correct_citation: {label_data.get('correct_citation', {})}")

## 4. Load Data → Training Format

**Strategy:** Với mỗi citation trong document:
- Lấy context theo CONTEXT_MODE: 'full' (cả đoạn) hoặc 'window_N' (N câu xung quanh marker)
- Với mỗi candidate paper: tạo 1 training example
  - input = [CLS] context [SEP] candidate title. candidate abstract [SEP]
  - label = 1 nếu candidate đúng, 0 nếu sai

**Ablation:** Đổi CONTEXT_MODE để chạy experiments khác nhau

In [ ]:
import json
import re
import random
from pathlib import Path
from datasets import Dataset

# ═══════════════════════════════════════════════════════════
# ⚙️ CONTEXT MODE — thay đổi giá trị này để chạy ablation
# ═══════════════════════════════════════════════════════════
# 'full'     → dùng toàn bộ đoạn text (baseline, có nhiễu)
# 'window_0' → chỉ câu chứa citation marker
# 'window_1' → câu chứa + 1 câu trước/sau
# 'window_2' → câu chứa + 2 câu trước/sau
CONTEXT_MODE = 'full'
# ═══════════════════════════════════════════════════════════


def get_context(text, citation_id, mode='full'):
    if mode == 'full':
        return text
    
    window = int(mode.split('_')[1])
    sentences = re.split(r'(?<=[.!?])\s+', text)
    
    target_idx = -1
    for i, sent in enumerate(sentences):
        if citation_id in sent:
            target_idx = i
            break
    
    if target_idx == -1:
        return text
    
    start = max(0, target_idx - window)
    end = min(len(sentences), target_idx + window + 1)
    return ' '.join(sentences[start:end])


def load_task2_data(data_dir, max_files=None, neg_ratio=3, context_mode='full'):
    data_path = Path(data_dir)
    label_files = sorted(data_path.glob('*.label'))
    
    if max_files:
        label_files = label_files[:max_files]
    
    total_files = len(label_files)
    print(f'📊 Loading {total_files:,} files | Mode: {context_mode}')
    
    examples = []
    skipped = 0
    stats = {'positive': 0, 'negative': 0}
    
    for file_idx, label_file in enumerate(label_files):
        if (file_idx + 1) % 1000 == 0:
            print(f'⏳ {file_idx+1:,}/{total_files:,} | Examples: {len(examples):,}')
        
        in_file = label_file.with_suffix('.in')
        
        try:
            with open(in_file) as f:
                in_data = json.load(f)
            with open(label_file) as f:
                label_data = json.load(f)
        except:
            skipped += 1
            continue
        
        text = in_data.get('text', '')
        if not text:
            skipped += 1
            continue
        
        candidates = in_data.get('citation_candidates', [])
        bib_entries = in_data.get('bib_entries', {})
        correct_citation = label_data.get('correct_citation', {})
        
        if not correct_citation or not candidates or not bib_entries:
            skipped += 1
            continue
        
        for citation_id, correct_paper_id in correct_citation.items():
            context = get_context(text, citation_id, mode=context_mode)
            
            if correct_paper_id in bib_entries:
                paper = bib_entries[correct_paper_id]
                paper_text = f"{paper.get('title', '')}. {paper.get('abstract', '')}"
                examples.append({'text_a': context, 'text_b': paper_text, 'label': 1})
                stats['positive'] += 1
            
            neg_candidates = [c for c in candidates if c != correct_paper_id and c in bib_entries]
            neg_sample = random.sample(neg_candidates, min(neg_ratio, len(neg_candidates)))
            
            for neg_paper_id in neg_sample:
                paper = bib_entries[neg_paper_id]
                paper_text = f"{paper.get('title', '')}. {paper.get('abstract', '')}"
                examples.append({'text_a': context, 'text_b': paper_text, 'label': 0})
                stats['negative'] += 1
    
    print(f'\n✅ {len(examples):,} examples | Pos: {stats["positive"]:,} | Neg: {stats["negative"]:,} | Skip: {skipped}')
    return examples


print('=' * 60)
print(f'CONTEXT MODE: {CONTEXT_MODE}')
print('=' * 60)

train_examples = load_task2_data(train_path, neg_ratio=3, context_mode=CONTEXT_MODE)
val_examples = load_task2_data(val_path, neg_ratio=3, context_mode=CONTEXT_MODE)

train_dataset = Dataset.from_list(train_examples)
val_dataset = Dataset.from_list(val_examples)

print(f'\n✅ Train: {len(train_dataset):,} | Val: {len(val_dataset):,}')

## 5. Tokenization

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Tokenizer: {MODEL_NAME}")


def tokenize_function(examples):
    return tokenizer(
        examples["text_a"],
        examples["text_b"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
    )


print("Tokenizing train...")
train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=["text_a", "text_b"])

print("Tokenizing val...")
val_tokenized = val_dataset.map(tokenize_function, batched=True, remove_columns=["text_a", "text_b"])

print(f"\n✅ Train: {len(train_tokenized):,} | Val: {len(val_tokenized):,}")

## 6. Load Model

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
print(f"✅ Model: {MODEL_NAME} (num_labels=2)")

## 7. Metrics

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", pos_label=1)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}


print("✅ Metrics defined")

## 8. Training Configuration

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import wandb

WANDB_PROJECT = "task2-citation-mapping"
CHECKPOINT_DIR = "/kaggle/working/checkpoints/task2_bert"
SAVE_DIR = "/kaggle/working/task2_bert_final"

try:
    if wandb.run is None:
        wandb.init(
            project=WANDB_PROJECT,
            name=f"bert-base-lr3e5-{CONTEXT_MODE}",
            config={"model": MODEL_NAME, "learning_rate": 3e-5, "max_steps": 5000,
                    "warmup_steps": 500, "context_mode": CONTEXT_MODE, "neg_ratio": 3},
            resume="allow",
        )
    report_to = "wandb"
    print("✅ Wandb initialized")
except:
    report_to = "none"
    print("⚠️ Wandb not available")


training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    max_steps=5000,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_dir="/kaggle/working/logs",
    logging_steps=100,
    report_to=report_to,
    fp16=True,
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"\n✅ Trainer ready | max_steps={training_args.max_steps} | LR={training_args.learning_rate} | batch=32")

## 9. Train

In [ ]:
print("=" * 60)
print("🚀 TRAINING BERT FOR CITATION MAPPING")
print("=" * 60)

trainer.train()

print("\n✅ Training complete!")

## 10. Evaluation

In [ ]:
print("📊 VALIDATION RESULTS")
print("=" * 60)

eval_results = trainer.evaluate()

for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

print("=" * 60)
print(f"\n✅ Accuracy:  {eval_results.get('eval_accuracy', 0):.2%}")
print(f"✅ Precision: {eval_results.get('eval_precision', 0):.2%}")
print(f"✅ Recall:    {eval_results.get('eval_recall', 0):.2%}")
print(f"✅ F1:        {eval_results.get('eval_f1', 0):.2%}")

## 11. Save Model & Report

In [ ]:
import os, shutil

os.makedirs(SAVE_DIR, exist_ok=True)
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
shutil.make_archive(SAVE_DIR, 'zip', SAVE_DIR)
print(f"✅ Model saved: {SAVE_DIR}")

summary = f"""============================================================
EXPERIMENT RESULTS - Task 2 Citation Mapping
============================================================
Model:        {MODEL_NAME}
Context:      {CONTEXT_MODE}
Max length:   {MAX_LENGTH}
Max steps:    {training_args.max_steps}
LR:           {training_args.learning_rate}
Batch:        {training_args.per_device_train_batch_size}x{training_args.gradient_accumulation_steps}={training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}
Warmup:       {training_args.warmup_steps}
Weight decay: {training_args.weight_decay}
Train data:   {len(train_tokenized):,}
Val data:     {len(val_tokenized):,}
------------------------------------------------------------
Accuracy:     {eval_results.get('eval_accuracy', 0):.4f}
Precision:    {eval_results.get('eval_precision', 0):.4f}
Recall:       {eval_results.get('eval_recall', 0):.4f}
F1:           {eval_results.get('eval_f1', 0):.4f}
Eval loss:    {eval_results.get('eval_loss', 0):.4f}
============================================================"""

print(summary)

os.makedirs("/kaggle/working/results", exist_ok=True)
with open("/kaggle/working/results/task2_report.txt", "w") as f:
    f.write(summary)

try:
    wandb.finish()
except:
    pass

print("\n🎉 TASK 2 TRAINING COMPLETE!")